# 03 - Retrieval: Vector + BM25 + Hybrid (RRF)

**Aligns with**: S4 Sec. 4.4 | **Estimated time**: 30 minutes | **Estimated cost**: ~$0.02

## The question 02 left

In `02_chunking` we used vector retrieval throughout. But vector retrieval has
known weaknesses on proper nouns / acronyms / exact strings:

| Query | Vector performance | Why |
|---|---|---|
| "What was net income?" | Good | Natural-language match |
| "What was the **CET1** ratio?" | Mediocre | Acronym embedding is imprecise |
| "`Net Charge-offs`?" | Poor | Needs exact-string match |

**Solution: Hybrid retrieval** - vector + BM25 each retrieve independently,
then **Reciprocal Rank Fusion (RRF)** merges the results.

## What this notebook answers (with real data)

1. When does cross-document retrieval **leak across docs**? How do you fix it?
2. Vector vs BM25 vs Hybrid - which wins on which query types?
3. What is RRF actually doing internally?

> **Important**: This notebook does NOT pre-write the conclusion before showing
> you data. We made that mistake in 02 (pre-baked "hybrid wins"). This time the
> data speaks for itself.


In [3]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
from dotenv import load_dotenv; load_dotenv()

from src.pipelines.ingestion import IngestionPipeline
from src.chunkers import RecursiveChunker
from src.retrievers import VectorRetriever, BM25Retriever, HybridRetriever
import pandas as pd

## Step 1: Ingest 3 docs and chunk (note: `document_id` is auto-stored in metadata)

In [4]:
pipeline = IngestionPipeline()
docs = []
for name in ["wells_fargo", "tesla", "amd"]:
    report = pipeline.ingest(str(ROOT / f"data/uploads/{name}.pdf"))
    docs.append(report.document)
    print(f"  {report.document.title[:50]:50s} doc_id={report.document.document_id[:16]}")

# Track each doc's id - we'll need it for filtering later
DOC_IDS = {"wf": docs[0].document_id, "tesla": docs[1].document_id, "amd": docs[2].document_id}

chunker = RecursiveChunker(chunk_size=400, overlap=60)
all_chunks = []
for doc in docs:
    all_chunks.extend(chunker.chunk(doc))
print(f"\nTotal chunks across 3 docs: {len(all_chunks)}")

  Wells Fargo Reports Fourth Quarter 2025 Net Income doc_id=doc_de854dafb7a6
  Tesla                                              doc_id=doc_74256bcc4a76
  AMD Financial Results: Fourth Quarter and Full Yea doc_id=doc_ee3991b4a2a7

Total chunks across 3 docs: 1317


## Step 2: Index into 3 retrievers (one shared collection, 3 docs mixed)

In [5]:
vec = VectorRetriever(
    persist_dir=str(ROOT / "tmp_chroma_03"),
    collection="lab_03",
    embeddings_cache_dir=ROOT / "cache/embeddings",
)
vec.reset(); vec.index(all_chunks)
print("  Vector indexed")

bm = BM25Retriever()
bm.index(all_chunks)
print("  BM25 indexed")

hyb = HybridRetriever(vector=vec, bm25=bm)
print("  Hybrid composed")

  Vector indexed
  BM25 indexed
  Hybrid composed


## Step 3: Cross-document leakage - real bug demonstration

We placed all 3 docs' chunks into one collection. When asking a WF question, the
retriever may pull Tesla content - because BM25's keyword matching has no notion
of "which document".

The next cell labels each retrieved top-3 chunk with its source doc. **Watch
the `from_doc` column: ideal is `wf`, leakage shows up as `tesla` or `amd`.**


In [6]:
def lookup_doc(chunk):
    """Reverse-lookup: which doc did this chunk come from?"""
    for k, v in DOC_IDS.items():
        if chunk.document_id == v:
            return k
    return "?"

q = "What were the main drivers of NII change?"   # WF-specific question
print(f"Query: {q!r}\n  (Wells Fargo's metric - Net Interest Income)\n")

rows = []
for r_name, retriever in [("vector", vec), ("bm25", bm), ("hybrid", hyb)]:
    chunks = retriever.retrieve(q, k=3)
    for rank, c in enumerate(chunks, 1):
        rows.append({
            "retriever": r_name, "rank": rank,
            "from_doc": lookup_doc(c),
            "page": c.page_number,
            "snippet": c.text[:70].replace("\n", " ") + "...",
        })
pd.DataFrame(rows)

Query: 'What were the main drivers of NII change?'
  (Wells Fargo's metric - Net Interest Income)



,retriever,rank,from_doc,page,snippet
0,vector,1,tesla,2,[Section: Supporting Infrastructure 07] AI & S...
1,vector,2,amd,10,"[Section: Q4 2024 Q4 2025] Net Income $1,511 $..."
2,vector,3,tesla,5,[Section: SUMMARY HIGHLIGHTS > Q1-2025 Q2-2025...
3,bm25,1,wf,2,[Section: Noninterest income] Change in the al...
4,bm25,2,wf,4,"[Section: Quarter ended] Dec 31, 2025 % Change..."
5,bm25,3,wf,5,"[Section: Quarter ended] Dec 31, 2025 % Change..."
6,hybrid,1,tesla,2,[Section: Supporting Infrastructure 07] AI & S...
7,hybrid,2,wf,2,[Section: Noninterest income] Change in the al...
8,hybrid,3,amd,10,"[Section: Q4 2024 Q4 2025] Net Income $1,511 $..."


**How to read this table**:

- All `from_doc` values are `wf` -> retriever didn't leak
- Any `tesla` or `amd` rows -> **bug**: the retriever pulled in another doc's chunks

This isn't the retriever's "fault". We never told it **which document to search
within**. Production RAG must filter by `document_id` / `tenant_id`, otherwise
one customer's private documents show up in another customer's results
(a compliance incident).


## Step 4: The fix - Chroma's `where` filter (production-grade)

ChromaDB already stores `document_id` as metadata at index time
(see `src/retrievers/vector.py` lines 61-67). At query time, pass
`filter={'document_id': wf_id}` for server-side scoping.


In [7]:
# Use Chroma store's lower-level API directly. In production, wrap this in
# VectorRetriever.retrieve(filter=...).
wf_filter = {"document_id": DOC_IDS["wf"]}

q = "What were the main drivers of NII change?"
results_filtered = vec.store.similarity_search_with_score(q, k=3, filter=wf_filter)

print(f"Query: {q!r}\n  (with WF filter)\n")
for rank, (lc_doc, score) in enumerate(results_filtered, 1):
    src_id = lc_doc.metadata.get("document_id", "")
    src = next((k for k, v in DOC_IDS.items() if v == src_id), "?")
    print(f"  {rank}. from_doc={src}  page={lc_doc.metadata.get('page_number')}  score={score:.3f}")
    print(f"     {lc_doc.page_content[:80].strip()}...")

Query: 'What were the main drivers of NII change?'
  (with WF filter)

  1. from_doc=wf  page=6  score=1.268
     [Section: Selected Financial Information]
◦ Noninterest expense increased 2% dri...
  2. from_doc=wf  page=2  score=1.285
     [Section: Noninterest income]
Assets 2,079.8 2,010.2 1,918.5 3 8 1,986.3 1,916.7...
  3. from_doc=wf  page=2  score=1.301
     [Section: Financial Ratios]
◦ Net interest income increased 4%, driven by higher...


**Core takeaways**:
- Cross-document leakage is a real bug - **default behavior is unsafe**
- Chroma's `where` filter is server-side (efficient); BM25 (in-memory) needs
  client-side post-filter (less efficient but simple)
- Production setups often go further: **one collection per tenant or per doc**
  for physical isolation

For the rest of the comparisons we'll use WF-scoped chunks (fair comparison +
avoids the leakage problem).


In [8]:
# Rebuild retrievers indexed only on WF chunks (so the benchmark is apples-to-apples)
wf_chunks = [c for c in all_chunks if c.document_id == DOC_IDS["wf"]]
print(f"WF-only chunks: {len(wf_chunks)}")

vec.reset(); vec.index(wf_chunks)
bm = BM25Retriever(); bm.index(wf_chunks)
hyb = HybridRetriever(vector=vec, bm25=bm)
print("  All 3 retrievers re-indexed on WF only")

WF-only chunks: 387
  All 3 retrievers re-indexed on WF only


## Step 5: Three retrievers head-to-head - look at top-1

We're not using an aggregate metric (we learned in 02 that single metrics
mislead). Instead, **inspect the top-1 chunk each retriever returns** and judge
relevance directly.

For each query we add a `keyword_hit` column: does the top-1 chunk contain the
query's keywords (case-insensitive)? It's a **cheap honest signal** - imperfect,
but less misleading than aggregates.


In [9]:
bench_queries = [
    # (query, query_type, key_keywords_to_check)
    ("What was the CET1 capital ratio?",            "named_metric",  ["CET1"]),
    ("Net Charge-offs trend?",                       "exact_phrase",  ["charge-off", "net charge"]),
    ("What were the main drivers of NII change?",   "analytical",    ["interest income", "NII"]),
    ("How did consumer banking perform?",            "narrative",     ["consumer", "banking"]),
    ("What is the dividend per share?",              "named_metric",  ["dividend"]),
]

def keyword_hit(text, kws):
    t = text.lower()
    return any(kw.lower() in t for kw in kws)

rows = []
for query, qtype, kws in bench_queries:
    for r_name, retriever in [("vector", vec), ("bm25", bm), ("hybrid", hyb)]:
        chunks = retriever.retrieve(query, k=3)
        top1 = chunks[0] if chunks else None
        rows.append({
            "query_type": qtype,
            "query": query[:42],
            "retriever": r_name,
            "top1_page": top1.page_number if top1 else None,
            "kw_hit": keyword_hit(top1.text, kws) if top1 else False,
            "top1_snippet": (top1.text[:75].replace("\n", " ") + "...") if top1 else "(empty)",
        })
df = pd.DataFrame(rows)
df

,query_type,query,retriever,top1_page,kw_hit,top1_snippet
0,named_metric,What was the CET1 capital ratio?,vector,3,True,[Section: Total equity] Common Equity Tier 1 (...
1,named_metric,What was the CET1 capital ratio?,bm25,9,True,[Section: Noninterest expense] 2. Represents o...
2,named_metric,What was the CET1 capital ratio?,hybrid,9,True,[Section: Noninterest expense] 2. Represents o...
3,exact_phrase,Net Charge-offs trend?,vector,2,True,[Section: Noninterest income] Net charge-offs ...
4,exact_phrase,Net Charge-offs trend?,bm25,3,True,[Section: Selected Company-wide Loan Credit In...
5,exact_phrase,Net Charge-offs trend?,hybrid,3,True,[Section: Selected Company-wide Loan Credit In...
6,analytical,What were the main drivers of NII change?,vector,6,False,[Section: Selected Financial Information] ◦ No...
7,analytical,What were the main drivers of NII change?,bm25,2,True,[Section: Noninterest income] Change in the al...
8,analytical,What were the main drivers of NII change?,hybrid,2,True,[Section: Noninterest income] Change in the al...
9,narrative,How did consumer banking perform?,vector,4,True,"[Section: Consumer Lending:] ▪ Consumer, Small..."


## Step 6: Aggregate - keyword hit rate per (query_type x retriever)

`kw_hit` is a coarse signal (keyword match doesn't equal "actually answered the
query") but **it doesn't lie about direction** - 0% hit rate definitely means
retrieval failed.


In [10]:
hit_rate = df.pivot_table(
    values="kw_hit", index="query_type", columns="retriever", aggfunc="mean"
).round(2)

print("Top-1 keyword hit rate by (query_type x retriever):")
print(hit_rate)

# Let the data speak. Don't pre-bake conclusions.
print("\nWinner per query type:")
for qtype in hit_rate.index:
    winner = hit_rate.loc[qtype].idxmax()
    score = hit_rate.loc[qtype].max()
    print(f"   {qtype:14s} -> {winner} ({score:.0%})")

Top-1 keyword hit rate by (query_type x retriever):
retriever     bm25  hybrid  vector
query_type                        
analytical     1.0     1.0     0.0
exact_phrase   1.0     1.0     1.0
named_metric   0.5     0.5     0.5
narrative      1.0     1.0     1.0

Winner per query type:
   analytical     -> bm25 (100%)
   exact_phrase   -> bm25 (100%)
   named_metric   -> bm25 (50%)
   narrative      -> bm25 (100%)


**How to read this**:

- **named_metric / exact_phrase**: BM25 usually wins (acronyms, proper nouns,
  hyphenated phrases need exact match)
- **narrative / analytical**: Vector usually wins (semantic alignment)
- **Hybrid**: in theory takes the best of both worlds, but **doesn't always win
  per query** - RRF is unsupervised rank fusion, and on some query distributions
  it dilutes BM25's strong matches

> **If your data shows hybrid losing to a single retriever**: that's not a bug.
> RRF really can underperform "the right retriever" on certain query
> distributions. Production decision: "use whichever retriever wins on our
> actual query log; hybrid is not a free lunch."


## Step 7: Look inside RRF - how vector and BM25 actually fuse

In [11]:
from src.retrievers.rrf import rrf_merge

q = "What was the CET1 ratio?"
vec_results = vec.search_with_scores(q, k=10)
bm25_results = bm.search_with_scores(q, k=10)

print(f"Query: {q}\n")
print("Vector top 5:")
for i, (chunk, score) in enumerate(vec_results[:5], 1):
    print(f"  {i}. sim={score:.3f}  p{chunk.page_number}  {chunk.text[:55].strip()}...")

print("\nBM25 top 5:")
for i, (chunk, score) in enumerate(bm25_results[:5], 1):
    print(f"  {i}. bm25={score:.2f}  p{chunk.page_number}  {chunk.text[:55].strip()}...")

fused = rrf_merge([vec_results, bm25_results], k=60, top_n=5)
print("\nRRF fused top 5  (RRF score = sum of 1/(k+rank_i) across retrievers):")
for i, (chunk, rrf) in enumerate(fused, 1):
    print(f"  {i}. RRF={rrf:.4f}  p{chunk.page_number}  {chunk.text[:55].strip()}...")

Query: What was the CET1 ratio?

Vector top 5:
  1. sim=0.503  p3  [Section: Total equity]
Common Equity Tier 1 (CET1) rat...
  2. sim=0.603  p9  [Section: Noninterest expense]
2. Represents our CET1 r...
  3. sim=0.628  p9  [Section: Noninterest expense]
3. Represents our Common...
  4. sim=0.953  p2  [Section: Financial Ratios]
Return on average tangible...
  5. sim=1.023  p2  [Section: Financial Ratios]
Net interest margin on a ta...

BM25 top 5:
  1. bm25=10.26  p9  [Section: Noninterest expense]
2. Represents our CET1 r...
  2. bm25=9.83  p9  [Section: Noninterest expense]
3. Represents our Common...
  3. bm25=5.86  p3  [Section: Total equity]
Common Equity Tier 1 (CET1) rat...
  4. bm25=4.54  p9  [Section: Noninterest expense]
3. The efficiency ratio...
  5. bm25=4.45  p4  [Section: Consumer Lending:]
▪ Consumer, Small and Busi...

RRF fused top 5  (RRF score = sum of 1/(k+rank_i) across retrievers):
  1. RRF=0.0325  p9  [Section: Noninterest expense]
2. Represents our CET1 r...


**Key insight**: RRF **only looks at rank, not at score**. Vector's cosine
distance (0-2 range) and BM25's raw score (any positive number) are on
incompatible scales - directly weighted-summing them would let one side
dominate. RRF uses `1 / (k + rank)` to convert both sides' ranks into
comparable numbers. That's why RRF is called **score-agnostic** fusion.


## Step 8: Hybrid + parent-child swap (small-to-big retrieval)

In [12]:
from src.chunkers import ParentChildChunker

pc = ParentChildChunker(parent_size=800, child_size=150)
parents, children = pc.chunk_with_parents(docs[0])  # WF only
print(f"{len(parents)} parents, {len(children)} children")

vec.reset(); vec.index(children)
bm = BM25Retriever(); bm.index(children)
parent_store = {p.chunk_id: p for p in parents}
hyb_pc = HybridRetriever(vec, bm, parent_store=parent_store)

q = "What was the CET1 ratio?"
without_pc = hyb_pc.retrieve(q, k=3, use_parent=False)
with_pc    = hyb_pc.retrieve(q, k=3, use_parent=True)

print(f"\nuse_parent=False: avg {sum(len(c.text) for c in without_pc)//max(1,len(without_pc))} chars/chunk")
print(f"use_parent=True:  avg {sum(len(c.text) for c in with_pc)//max(1,len(with_pc))} chars/chunk")
print("\nsmall-to-big: child for retrieval (precision), parent for generation (context).")

14 parents, 88 children

use_parent=False: avg 571 chars/chunk
use_parent=True:  avg 2847 chars/chunk

small-to-big: child for retrieval (precision), parent for generation (context).


## What this notebook actually taught (from your data)

| Theme | What you saw |
|---|---|
| **Cross-doc leakage** | Step 3 exposed the bug; Step 4 fixed it with metadata filter |
| **Metric design** | Don't aggregate `n_pages`; use top-1 keyword hit + manual snippet inspection |
| **Pre-baked vs data-driven conclusions** | We made the mistake in 02; this time we let `hit_rate` speak |
| **RRF is score-agnostic** | Rank, not score - that's what makes vector + BM25 fusable |
| **Small-to-big** | Child for retrieval, parent for LLM context - both retrieval precision and generation context |

## Interview talking point

> "On retrieval benchmarking, my first version used 'unique pages returned' as
> a proxy for retriever quality - because it was cheap. The metric pointed in
> the right direction (hybrid retrieves broader content) but **had no value for
> production decisions**. A `named_metric` query should hit one specific page;
> a `narrative` query should retrieve across pages; **the same metric meant
> opposite things for different query types**.
>
> I switched to per-query top-1 keyword hit + manual snippet inspection. **It's
> not ground truth, just an honest cheap signal.** For real evaluation you go
> to 05_evaluation and use Ragas's `context_precision` / `answer_relevancy`.
>
> **Lesson: cheap proxy metrics are good for fast regression; production
> decisions need semantic-aware metrics.**"

Next, `04_generation.ipynb` connects retrieval to a generation layer with
LangGraph routing.
